# Preprocessing / Noise removal Investigation

Preprocessing:
- Resample (to matched rate)
- BPF
- Break in to 8s windows with 2s stride
- Per window z-score norm

Noise Removal:
- NLMS Linear adaptive filter

## Preprocessing

In [1]:
# Imports

import numpy as np
import matplotlib.pyplot as plt
import pickle as pkl
import os
from scipy.signal import butter, sosfiltfilt, decimate, resample_poly
from math import gcd

%matplotlib widget

plt.close('all')

In [2]:
# Preprocessing constants

DATA_ROOT = "/Volumes/LPM02 storage/Datasets/Bio/DaLiA/data"
FS = 32
FS_PPG_RAW = 64  # Hz — original PPG sample rate before decimation
FS_ACC_RAW = 32  # Hz — original ACC sample rate (no change)
BPF_ORDER = 4
BPF_FC1 = 0.4
BPF_FC2 = 4.0
T_WIN = 8.0
T_STRIDE = 2.0
N_WIN = int(T_WIN * FS)       # samples per window: 256
N_STRIDE = int(T_STRIDE * FS) # samples per stride: 64
N_SUBJECTS = 15

In [ ]:
# Load raw data for all subjects

data_raw = {}

for s in range(1, N_SUBJECTS + 1):
    subj_id = f"S{s}"
    path = os.path.join(DATA_ROOT, subj_id, f"{subj_id}.pkl")
    with open(path, 'rb') as f:
        d = pkl.load(f, encoding='latin1')
    data_raw[subj_id] = {
        'ppg': d['signal']['wrist']['BVP'],
        'acc': d['signal']['wrist']['ACC'],
        'label': d['label'],
    }
    print(f"{subj_id} loaded — PPG: {data_raw[subj_id]['ppg'].shape}, "
          f"ACC: {data_raw[subj_id]['acc'].shape}, "
          f"labels: {data_raw[subj_id]['label'].shape}")

In [ ]:
# Preprocess (and window) all subjects

from scipy.signal import resample_poly
from math import gcd

def get_resample_factors(fs_original, fs_target):
    """Compute up/down factors for resample_poly given original and target sample rates."""
    g = gcd(fs_original, fs_target)
    return fs_target // g, fs_original // g

sos = butter(BPF_ORDER, [BPF_FC1, BPF_FC2], btype='bandpass', fs=FS, output='sos')

up_ppg, down_ppg = get_resample_factors(FS_PPG_RAW, FS)
up_acc, down_acc = get_resample_factors(FS_ACC_RAW, FS)

print(f"PPG resample factors: up={up_ppg}, down={down_ppg} ({FS_PPG_RAW} -> {FS} Hz)")
print(f"ACC resample factors: up={up_acc}, down={down_acc} ({FS_ACC_RAW} -> {FS} Hz)")

data_preproc = {}

for subj_id, d in data_raw.items():
    # Resample to target FS
    ppg = resample_poly(d['ppg'], up_ppg, down_ppg, axis=0)
    if (up_acc, down_acc) != (1, 1):
        acc = resample_poly(d['acc'], up_acc, down_acc, axis=0)
    else:
        acc = d['acc'].copy()

    # BPF — identical filter applied to both at target FS
    ppg = sosfiltfilt(sos, ppg, axis=0)
    acc = sosfiltfilt(sos, acc, axis=0)

    # Window and normalize
    n_samples = len(ppg)
    n_windows = (n_samples - N_WIN) // N_STRIDE + 1

    ppg_wins = np.zeros((n_windows, N_WIN, 1))
    acc_wins = np.zeros((n_windows, N_WIN, 3))

    for i in range(n_windows):
        start = i * N_STRIDE
        end   = start + N_WIN

        p = ppg[start:end]
        ppg_wins[i] = (p - np.mean(p)) / np.std(p)

        a = acc[start:end]
        acc_wins[i] = (a - np.mean(a, axis=0)) / np.std(a, axis=0)

    data_preproc[subj_id] = {
        'ppg':   ppg_wins,
        'acc':   acc_wins,
        'label': d['label'][:n_windows],
    }
    print(f"{subj_id} — windows: {n_windows}, labels: {len(d['label'][:n_windows])}")